# Notebook 04 — Forward Kinematics

This notebook translates and updates the MATLAB live script **04_rvc1_ch7.1_FwdKins.mlx** into Python.

The progression is:

1. Forward kinematics of a simple 1-DoF planar arm
2. Elementary Transform Sequences (ETS) in 2D
3. 2-DoF and 3-DoF planar manipulators
4. Symbolic planar forward kinematics
5. 3D ETS models
6. Denavit–Hartenberg (DH) robot models
7. Built-in robot models: Puma 560 and Franka Emika Panda
8. Base and tool transformations
9. Robots as kinematic trees
10. Link/frame introspection
11. Joint-space trajectories and forward kinematics along a trajectory

The examples use **Robotics Toolbox for Python** together with **Spatial Math Toolbox for Python**.

## API version note

This notebook was updated against the current Python APIs in September 2026.

The principal MATLAB → Python changes used here are:

| MATLAB-style concept | Current Python equivalent |
|---|---|
| `ETS2.Rz('q1')` | `ET2.R()` |
| `ETS2.Tx(a1)` | `ET2.tx(a1)` |
| `ETS3.Rz(q1)` | `ET.Rz()` |
| `ETS3.Tz(a1)` | `ET.tz(a1)` |
| `SerialLink(...)` | `DHRobot(...)` or `Robot(ets)` |
| `Revolute(...)` | `RevoluteDH(...)` for a DH model |
| `mdl_puma560` | `rtb.models.DH.Puma560()` |
| MATLAB `rigidBodyTree` | Python `Robot` / URDF robot tree |
| MATLAB `loadrobot(...)` | models such as `rtb.models.URDF.Panda()` |
| `jtraj(...)` matrix output | a `Trajectory` object; positions are in `.q` |

A particularly important Python convention is that **pose objects** such as `SE3` compose with `*`, whereas raw NumPy transformation matrices compose with `@`.

### Installation

If the packages are not already installed, run the following once in your Python environment.

```bash
pip install -U "roboticstoolbox-python[swift]" spatialmath-python
```

The core toolbox does not require Swift, but the optional `swift` extra provides a higher-quality web-based robot visualizer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
import importlib.metadata as metadata

import roboticstoolbox as rtb
from roboticstoolbox import ET, ET2, Robot, DHRobot, RevoluteDH, jtraj
from spatialmath import SE2, SE3
from spatialmath.base import trot2, transl2

np.set_printoptions(precision=4, suppress=True)

In [ ]:
# Record installed versions for reproducibility.
print("roboticstoolbox-python:", metadata.version("roboticstoolbox-python"))
print("spatialmath-python:    ", metadata.version("spatialmath-python"))
print("numpy:                 ", np.__version__)

---
# 1. Forward kinematics of a very simple 1-DoF robot

A serial-link manipulator is built from links connected by joints. Forward kinematics answers:

> **Given the joint coordinates, where is the end effector?**

Consider a single planar revolute joint with link length \(a_1\) and joint angle \(q_1\). The end-effector pose is

\[
{}^0T_E = R(q_1)\,T_x(a_1).
\]

Rotation is applied first, and the translation then occurs along the **rotated local \(x\)-axis**.

In [ ]:
a1 = 1.0       # link length [m]
q1 = 0.2       # joint angle [rad]

T1_array = trot2(q1) @ transl2(a1, 0)
T1_array

The result above is a raw \(3\times3\) homogeneous transformation matrix in \(SE(2)\). We can wrap it in an `SE2` object to access rotation and translation directly.

In [ ]:
T1 = SE2(T1_array)
print(T1)

In [ ]:
print("Rotation:")
print(T1.R)

print("\nTranslation:")
print(T1.t)

For this one-link arm,

\[
x_E = a_1\cos q_1,\qquad
y_E = a_1\sin q_1.
\]

The next cell is both an example and a numerical self-check.

In [ ]:
expected_position = np.array([
    a1 * np.cos(q1),
    a1 * np.sin(q1)
])

print("FK position:      ", T1.t)
print("analytic position:", expected_position)

assert np.allclose(T1.t, expected_position)

---
# 2. Elementary Transform Sequences (ETS) in 2D

An **Elementary Transform Sequence** describes robot kinematics as an ordered sequence of simple rotations and translations.

For the same one-joint arm,

\[
\xi_E(q) = R(q_1)\oplus T_x(a_1).
\]

In the current Python API:

- `ET2.R()` with no angle represents a **revolute joint variable**.
- `ET2.tx(a1)` represents a **constant translation**.

In [ ]:
ets1 = ET2.R() * ET2.tx(a1)
ets1

In [ ]:
print("Number of joints:", ets1.n)
print("Joint structure: ", ets1.structure)

`fkine(q)` substitutes numerical joint coordinates into the sequence and returns an `SE2` pose.

In [ ]:
for angle in [0, np.pi/6, np.pi/4, np.pi/3, np.pi/2]:
    T = ets1.fkine([angle])
    print(f"q1 = {np.rad2deg(angle):5.1f} deg  ->  position = {T.t}")

### Cross-check ETS against the direct homogeneous-transform calculation

In [ ]:
T_direct = SE2(trot2(q1) @ transl2(a1, 0))
T_ets = ets1.fkine([q1])

print("Direct:\n", T_direct)
print("\nETS:\n", T_ets)

assert np.allclose(T_direct.A, T_ets.A)

---
# 3. A 2-DoF planar robot

For two revolute joints and two links,

\[
{}^0T_E =
R(q_1)T_x(a_1)
R(q_2)T_x(a_2).
\]

The order matters: each link translation is performed in the coordinate frame produced by the transformations to its left.

In [ ]:
a1 = 1.0
a2 = 1.0

ets2 = (
    ET2.R() * ET2.tx(a1)
    * ET2.R() * ET2.tx(a2)
)

ets2

In [ ]:
print("Number of joints:", ets2.n)
print("Structure:", ets2.structure)

In [ ]:
test_configs = np.array([
    [0, 0],
    [np.pi/2, 0],
    [0, np.pi/2],
    [np.pi/2, np.pi/2],
])

for q in test_configs:
    T = ets2.fkine(q)
    print(f"q = {np.rad2deg(q)} deg -> pE = {T.t}")

For a planar 2R arm,

\[
x_E = a_1\cos q_1 + a_2\cos(q_1+q_2),
\]

\[
y_E = a_1\sin q_1 + a_2\sin(q_1+q_2).
\]

We can use this expression to verify the ETS result.

In [ ]:
q = np.deg2rad([30, 40])

T = ets2.fkine(q)

p_expected = np.array([
    a1*np.cos(q[0]) + a2*np.cos(q[0] + q[1]),
    a1*np.sin(q[0]) + a2*np.sin(q[0] + q[1])
])

print("ETS:", T.t)
print("analytic:", p_expected)

assert np.allclose(T.t, p_expected)

### Visualizing a planar robot

For simple planar arms it is often clearer to draw the joint locations directly. This helper uses only NumPy and Matplotlib.

In [ ]:
def planar_joint_positions(q, lengths):
    # Return base and successive joint/end-effector positions for a planar revolute chain.
    q = np.asarray(q, dtype=float)
    lengths = np.asarray(lengths, dtype=float)

    theta = np.cumsum(q)
    dx = lengths * np.cos(theta)
    dy = lengths * np.sin(theta)

    x = np.r_[0.0, np.cumsum(dx)]
    y = np.r_[0.0, np.cumsum(dy)]
    return x, y


def plot_planar_arm(q, lengths, title=None):
    x, y = planar_joint_positions(q, lengths)

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(x, y, "o-", linewidth=2)
    ax.axhline(0, linewidth=0.5)
    ax.axvline(0, linewidth=0.5)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True)
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    if title:
        ax.set_title(title)
    reach = sum(lengths)
    ax.set_xlim(-reach - 0.2, reach + 0.2)
    ax.set_ylim(-reach - 0.2, reach + 0.2)
    return ax

In [ ]:
q = np.deg2rad([90, 45])
plot_planar_arm(q, [a1, a2], "2R planar arm")
plt.show()

---
# 4. A 3-DoF planar robot

Adding a third revolute joint extends the same transform sequence.

In [ ]:
a1 = a2 = a3 = 1.0

ets3 = (
    ET2.R() * ET2.tx(a1)
    * ET2.R() * ET2.tx(a2)
    * ET2.R() * ET2.tx(a3)
)

print(ets3)
print("structure:", ets3.structure)

In [ ]:
q = np.array([0, np.pi/2, 0])
T = ets3.fkine(q)

print(T)
print("end-effector position:", T.t)

In [ ]:
plot_planar_arm(q, [a1, a2, a3], "3R planar arm")
plt.show()

## Symbolic forward kinematics for the planar 3R arm

The original MATLAB notebook also develops a symbolic result. Here we use SymPy directly, which keeps the algebra explicit and avoids dependence on toolbox-specific symbolic ETS behavior.

In [ ]:
q1_s, q2_s, q3_s = sp.symbols("q1 q2 q3", real=True)
a1_s, a2_s, a3_s = sp.symbols("a1 a2 a3", positive=True, real=True)

theta_s = q1_s + q2_s + q3_s

x_s = (
    a1_s * sp.cos(q1_s)
    + a2_s * sp.cos(q1_s + q2_s)
    + a3_s * sp.cos(theta_s)
)

y_s = (
    a1_s * sp.sin(q1_s)
    + a2_s * sp.sin(q1_s + q2_s)
    + a3_s * sp.sin(theta_s)
)

T3_symbolic = sp.Matrix([
    [sp.cos(theta_s), -sp.sin(theta_s), x_s],
    [sp.sin(theta_s),  sp.cos(theta_s), y_s],
    [0,                0,               1],
])

T3_symbolic

The symbolic expression makes two patterns easy to see:

- the end-effector orientation is \(q_1+q_2+q_3\);
- the position is the vector sum of the three links after their cumulative rotations.

---
# 5. Extending ETS to a 3D robot

The 3D class is `ET`. It provides variable rotations `Rx()`, `Ry()`, `Rz()` and translations `tx()`, `ty()`, `tz()`.

A transform with an unspecified parameter, such as `ET.Rz()`, is a joint variable.

In [ ]:
a1, a2, a3, a4 = 0.20, 0.20, 0.20, 0.10

ets_3d = (
    ET.Rz()
    * ET.tz(a1)
    * ET.Ry()
    * ET.tz(a2)
    * ET.Ry()
    * ET.tz(a3)
    * ET.Ry()
    * ET.tz(a4)
)

ets_3d

In [ ]:
print("Number of joints:", ets_3d.n)
print("Joint structure: ", ets_3d.structure)

In [ ]:
q = np.deg2rad([20, 30, -40, 15])
T = ets_3d.fkine(q)

print(T)
print("translation:", T.t)
print("rotation:\n", T.R)

An ETS can also be promoted to a full `Robot` object. The constructor partitions the transform sequence into links and joints.

In [ ]:
robot_ets = Robot(ets_3d, name="4-DoF ETS arm")
print(robot_ets)

In [ ]:
robot_ets.plot(q)

For an interactive teach pendant, use:

```python
robot_ets.teach(q)
```

It is left as an optional command because interactive GUI calls can block a notebook kernel.

---
# 6. Denavit–Hartenberg parameterization

For a standard DH link,

\[
{}^{j-1}T_j =
R_z(\theta_j)\,
T_z(d_j)\,
T_x(a_j)\,
R_x(\alpha_j).
\]

The current Python Toolbox represents a standard revolute DH link with `RevoluteDH`.

In [ ]:
L = RevoluteDH(a=1.0)
print(L)
print("link length a =", L.a)

In [ ]:
q1 = 0.5
A1 = L.A(q1)

print(A1)
print("type:", type(A1))

`L.A(q)` returns the relative `SE3` transform contributed by one DH link at joint coordinate \(q\).

## Constructing a 2-DoF DH robot

The MATLAB `SerialLink([Revolute(...) ...])` pattern is represented in current Python by `DHRobot([...])`.

In [ ]:
robot_dh = DHRobot(
    [
        RevoluteDH(a=1.0),
        RevoluteDH(a=1.0),
    ],
    name="2R DH robot"
)

print(robot_dh)

In [ ]:
rng = np.random.default_rng(7)
q = rng.uniform(0, np.pi/2, size=2)

print("q [rad] =", q)
print(robot_dh.fkine(q))

In [ ]:
print("Joint structure:", robot_dh.structure)
print("Equivalent ETS:")
print(robot_dh.ets())

In [ ]:
robot_dh.plot(q)

---
# 7. Built-in robot models

Robotics Toolbox for Python contains DH, ETS, and URDF robot models. The model catalog can be displayed with:

```python
from roboticstoolbox.models.catalog import catalog
catalog()
```

Below we use the **Puma 560** and **Franka Emika Panda**.

## Puma 560

In [ ]:
puma = rtb.models.DH.Puma560()
print(puma)

The Puma model provides named joint configurations that can be passed directly to `fkine`.

In [ ]:
print("qz — zero:\n", puma.qz)
print("\nqr — ready:\n", puma.qr)
print("\nqn — nominal:\n", puma.qn)
print("\nqs — stretched:\n", puma.qs)

In [ ]:
T_zero = puma.fkine(puma.qz)
T_nominal = puma.fkine(puma.qn)

print("FK at qz:")
print(T_zero)

print("\nFK at qn:")
print(T_nominal)

In [ ]:
print("Nominal end-effector position:", T_nominal.t)
print("Nominal end-effector rotation:\n", T_nominal.R)

In [ ]:
puma.plot(puma.qn)

The current Python `teach` method is available for robot models as well:

```python
puma.teach(puma.qn)
```

## Franka Emika Panda

For a link-tree and mesh representation, the URDF model is particularly useful.

In [ ]:
panda = rtb.models.URDF.Panda()
print(panda)

In [ ]:
print("Number of actuated arm joints:", panda.n)
print("Structure:", panda.structure)
print("\nReady configuration:")
print(panda.qr)

In [ ]:
T_panda_ready = panda.fkine(panda.qr)
print(T_panda_ready)

Forward kinematics is not limited to the end effector. On a tree robot we can request the pose of a specific link.

In [ ]:
T_link3 = panda.fkine(panda.qz, end="panda_link3")
print(T_link3)

---
# 8. Base and tool transformations

A robot model has:

- a **base transform**, locating the robot relative to the world frame;
- a **tool transform**, locating a tool frame relative to the nominal final link.

The complete mapping is

\[
{}^WT_E =
{}^WT_0\;
{}^0T_N(q)\;
{}^NT_E.
\]

In [ ]:
puma_bt = rtb.models.DH.Puma560()
q = puma_bt.qn.copy()

T_original = puma_bt.fkine(q)
print("Original pose:")
print(T_original)

## Base transformation

Place the entire robot on a \(0.75\,\text{m}\) pedestal.

In [ ]:
puma_bt.base = SE3(0, 0, 0.75)

T_with_base = puma_bt.fkine(q)
print(T_with_base)

In [ ]:
print("Position before:", T_original.t)
print("Position after: ", T_with_base.t)
print("Difference:     ", T_with_base.t - T_original.t)

assert np.allclose(
    T_with_base.t - T_original.t,
    np.array([0.0, 0.0, 0.75])
)

## Tool transformation

Attach a tool whose origin is \(0.10\,\text{m}\) along the end-effector's local \(x\)-axis.

In [ ]:
puma_bt.tool = SE3(0.10, 0, 0)

T_with_tool = puma_bt.fkine(q)
print(T_with_tool)

The tool translation is defined in the **local end-effector frame**, not directly in world coordinates. Its world-coordinate displacement is therefore the current end-effector rotation applied to the local tool offset.

In [ ]:
actual_tool_shift = T_with_tool.t - T_with_base.t
expected_tool_shift = T_with_base.R @ np.array([0.10, 0.0, 0.0])

print("actual shift:  ", actual_tool_shift)
print("expected shift:", expected_tool_shift)

assert np.allclose(actual_tool_shift, expected_tool_shift)

---
# 9. Robots as kinematic trees

The original MATLAB live script next introduces `rigidBodyTree`. Robotics Toolbox for Python expresses the same core idea with `Robot` objects built from links, ETS descriptions, or URDF files.

A serial chain is a special case of a tree:

\[
\text{base}\rightarrow\text{link}_1\rightarrow\text{link}_2\rightarrow\cdots\rightarrow\text{end effector}.
\]

## Create a simple robot directly from a 3D ETS

Although this robot moves only in the \(xy\)-plane, representing it with 3D transforms lets us use the standard `Robot` tree infrastructure.

In [ ]:
planar_ets_3d = (
    ET.Rz()
    * ET.tx(1.0)
    * ET.Rz()
    * ET.tx(1.0)
)

planar_robot = Robot(planar_ets_3d, name="2R planar tree")
print(planar_robot)

In [ ]:
q = np.deg2rad([30, 40])
T = planar_robot.fkine(q)

print(T)
print("end-effector position:", T.t)

### Poses of all link frames

In [ ]:
all_frames = planar_robot.fkine_all(q)

for i, Ti in enumerate(all_frames):
    print(f"frame {i}: translation = {Ti.t}")

This is the Python analogue of querying transforms to intermediate rigid bodies in a tree model.

---
# 10. Robot introspection

For a robot that you did not create yourself, inspect the link hierarchy before making assumptions about frame names, parent-child relationships, or end effectors.

In [ ]:
panda = rtb.models.URDF.Panda()
panda.hierarchy()

Links can be retrieved by name.

In [ ]:
link2 = panda["panda_link2"]

print("link:")
print(link2)

print("\nparent:")
print(link2.parent)

print("\nchildren:")
print(link2.children)

print("\nvariable transform:")
print(link2.v)

The robot can also expose the ETS from its base to the default end effector.

In [ ]:
panda_ets = panda.ets()
print(panda_ets)

---
# 11. Forward kinematics along a joint trajectory

A trajectory is a sequence of joint configurations \(q_k\). Applying forward kinematics to every row produces the corresponding end-effector path.

In [ ]:
puma_traj = rtb.models.DH.Puma560()

traj = jtraj(puma_traj.qz, puma_traj.qr, 100)

print(traj)
print("joint trajectory shape:", traj.q.shape)

In [ ]:
T_path = puma_traj.fkine(traj.q)

print("number of poses:", len(T_path))
print("start position:", T_path[0].t)
print("final position:", T_path[-1].t)

A basic Cartesian visualization of the resulting end-effector path:

In [ ]:
xyz = np.array([T.t for T in T_path])

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection="3d")
ax.plot(xyz[:, 0], xyz[:, 1], xyz[:, 2])
ax.scatter(*xyz[0], label="start")
ax.scatter(*xyz[-1], label="finish")
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.set_zlabel("z [m]")
ax.set_title("Puma 560 end-effector path")
ax.legend()
plt.show()

To animate the robot itself, the same trajectory can be passed to a plotting backend:

```python
puma_traj.plot(traj.q, backend="pyplot")
```

or, with Swift installed:

```python
puma_traj.plot(traj.q, backend="swift")
```

---
# 12. Consistency checks

These checks make several key equivalences explicit:

1. direct planar transforms and ETS give the same pose;
2. the 2R ETS agrees with the analytic planar solution;
3. the robot base contributes a world-frame translation;
4. the tool displacement is rotated from the local tool frame into the world frame;
5. vectorized forward kinematics starts and ends at the same poses obtained from individual configurations.

In [ ]:
# 1. 1R direct transform vs ETS
a1 = 1.0
q1 = 0.37
T_direct = SE2(trot2(q1) @ transl2(a1, 0))
T_ets = (ET2.R() * ET2.tx(a1)).fkine([q1])
assert np.allclose(T_direct.A, T_ets.A)

# 2. 2R ETS vs analytic position
a1, a2 = 0.7, 1.1
q = np.array([0.4, -0.25])
e = ET2.R() * ET2.tx(a1) * ET2.R() * ET2.tx(a2)
T = e.fkine(q)
p = np.array([
    a1*np.cos(q[0]) + a2*np.cos(q.sum()),
    a1*np.sin(q[0]) + a2*np.sin(q.sum())
])
assert np.allclose(T.t, p)

# 3 and 4 are asserted in the base/tool section.

# 5. trajectory endpoints
assert np.allclose(T_path[0].A, puma_traj.fkine(puma_traj.qz).A)
assert np.allclose(T_path[-1].A, puma_traj.fkine(puma_traj.qr).A)

print("All notebook consistency checks passed.")

---
# 13. MATLAB → Python reference

| MATLAB live-script operation | Python used here |
|---|---|
| `trot2(q)*transl2(a,0)` | `trot2(q) @ transl2(a, 0)` |
| `SE2(...)` | `SE2(...)` |
| `Rz('q1')*Tx(a1)` in ETS2 | `ET2.R() * ET2.tx(a1)` |
| `E.fkine(q)` | `ets.fkine(q)` |
| `ETS3.Rz(...)` | `ET.Rz()` |
| `ETS3.Tz(a)` | `ET.tz(a)` |
| `Revolute('a',1)` | `RevoluteDH(a=1)` |
| `SerialLink([...])` | `DHRobot([...])` |
| `mdl_puma560` | `rtb.models.DH.Puma560()` |
| `robot.base = ...` | `robot.base = SE3(...)` |
| `robot.tool = ...` | `robot.tool = SE3(...)` |
| MATLAB `rigidBodyTree` | Python `Robot` / URDF tree |
| `getTransform(...)` | `fkine(..., start=..., end=...)` or `fkine_all(...)` |
| `showdetails` | `print(robot)`, `hierarchy()`, `showgraph()` |
| `loadrobot("frankaEmikaPanda")` | `rtb.models.URDF.Panda()` |
| `jtraj(q0,q1,N)` | `jtraj(q0,q1,N)` and use `traj.q` |

## Main idea

Forward kinematics is always a **composition of relative link transformations**. ETS, DH, and URDF/tree models differ in how they describe those transformations; once the model is built, the goal is the same:

\[
q \longmapsto {}^0T_E(q).
\]